In [1]:
import pandas as pd
from faker import Faker
import random
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly_resampler import FigureResampler, register_plotly_resampler

In [2]:
df = pd.read_csv('pcap2ipfix-applabel-yafscii.csv')
df

,start-time,end-time,duration,rtt,proto,sip,sp,dip,dp,iflags,...,tag,rtag,pkt,oct,rpkt,roct,app,entropy,rentropy,end-reason
0,2024-06-19 20:06:58.527,2024-06-19 20:06:58.552,0.025,0.000,6,212.102.59.160,8080,192.168.50.2,59002,AP,...,0,0,4,232,3,156,0,0,0,NaN
1,2024-06-19 20:06:58.552,2024-06-19 20:06:58.552,0.000,0.000,6,212.102.59.160,8080,192.168.50.2,59002,A,...,0,0,1,52,1,40,0,0,0,NaN
2,2024-06-19 20:06:58.657,2024-06-19 20:06:58.676,0.019,0.000,6,181.214.164.4,443,192.168.50.2,59001,AP,...,0,0,3,180,3,156,0,0,0,NaN
3,2024-06-19 20:07:04.082,2024-06-19 20:07:04.388,0.306,0.035,6,192.168.50.2,59019,129.137.3.13,443,S,...,0,0,13,1062,11,6859,443,223,241,NaN
4,2024-06-19 20:07:17.577,2024-06-19 20:07:17.591,0.014,0.000,6,13.107.253.40,443,192.168.50.2,58993,AP,...,0,0,5,323,4,208,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60223,2024-06-20 00:46:22.892,2024-06-20 00:50:12.469,229.577,0.000,17,192.168.50.145,5353,224.0.0.251,5353,0,...,0,0,19,2100,0,0,0,132,0,eof
60224,2024-06-20 00:46:22.892,2024-06-20 00:50:12.470,229.578,0.000,17,fe80::005b:ef11:02da:d960,5353,ff02::00fb,5353,0,...,0,0,18,2336,0,0,0,132,0,eof
60225,2024-06-20 00:50:05.473,2024-06-20 00:50:12.601,7.128,0.020,17,192.168.50.2,65356,142.251.111.93,443,0,...,0,0,212,76154,383,325897,51443,250,251,eof
60226,2024-06-20 00:38:12.804,2024-06-20 00:50:12.981,720.177,0.000,17,192.168.50.42,9487,255.255.255.255,9478,0,...,0,0,13,884,0,0,0,99,0,eof


In [3]:
def double_frame(input_df):
    frame = input_df
    copy_frame = frame.copy()

    frame['start-time'] = pd.to_datetime(frame['start-time'])
    frame['end-time'] = pd.to_datetime(frame['end-time'])

    random_seconds = np.random.randint(86401, 172802, size=len(copy_frame))
    random_offsets = pd.to_timedelta(random_seconds, unit='s')
    copy_frame['start-time'] = frame['start-time'] + (frame['end-time'].max() - frame['start-time'].min()) + random_offsets
    copy_frame['end-time'] = frame['end-time'] + (frame['end-time'].max() - frame['start-time'].min()) + random_offsets

    random_values = np.random.randint(1, 5001, size=len(copy_frame))
    copy_frame['oct'] = copy_frame['oct'] + random_values
    copy_frame['roct'] = copy_frame['roct'] + random_values
    copy_frame['pkt'] = copy_frame['pkt'] + random_values
    copy_frame['rpkt'] = copy_frame['rpkt'] + random_values
    
    frame_combined = pd.concat([frame, copy_frame])
    frame_combined.reset_index(drop=True, inplace=True)
    return frame_combined


In [4]:
for i in range(5):
    df = double_frame(df)
df

,start-time,end-time,duration,rtt,proto,sip,sp,dip,dp,iflags,...,tag,rtag,pkt,oct,rpkt,roct,app,entropy,rentropy,end-reason
0,2024-06-19 20:06:58.527,2024-06-19 20:06:58.552,0.025,0.000,6,212.102.59.160,8080,192.168.50.2,59002,AP,...,0,0,4,232,3,156,0,0,0,NaN
1,2024-06-19 20:06:58.552,2024-06-19 20:06:58.552,0.000,0.000,6,212.102.59.160,8080,192.168.50.2,59002,A,...,0,0,1,52,1,40,0,0,0,NaN
2,2024-06-19 20:06:58.657,2024-06-19 20:06:58.676,0.019,0.000,6,181.214.164.4,443,192.168.50.2,59001,AP,...,0,0,3,180,3,156,0,0,0,NaN
3,2024-06-19 20:07:04.082,2024-06-19 20:07:04.388,0.306,0.035,6,192.168.50.2,59019,129.137.3.13,443,S,...,0,0,13,1062,11,6859,443,223,241,NaN
4,2024-06-19 20:07:17.577,2024-06-19 20:07:17.591,0.014,0.000,6,13.107.253.40,443,192.168.50.2,58993,AP,...,0,0,5,323,4,208,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1927291,2024-08-23 17:30:42.896,2024-08-23 17:34:32.473,229.577,0.000,17,192.168.50.145,5353,224.0.0.251,5353,0,...,0,0,12155,14236,12136,12136,0,132,0,eof
1927292,2024-08-25 01:41:18.896,2024-08-25 01:45:08.474,229.578,0.000,17,fe80::005b:ef11:02da:d960,5353,ff02::00fb,5353,0,...,0,0,16577,18895,16559,16559,0,132,0,eof
1927293,2024-08-23 08:32:22.477,2024-08-23 08:32:29.605,7.128,0.020,17,192.168.50.2,65356,142.251.111.93,443,0,...,0,0,15809,91751,15980,341494,51443,250,251,eof
1927294,2024-08-24 03:29:25.808,2024-08-24 03:41:25.985,720.177,0.000,17,192.168.50.42,9487,255.255.255.255,9478,0,...,0,0,13689,14560,13676,13676,0,99,0,eof


In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['pkt'], mode='markers', name='oct'))
fig_resampled = FigureResampler(fig)
#fig_resampled.show()

In [6]:
#iflags_group = df.groupby('iflags').size().reset_index(name = 'count')
#fig = FigureResampler(px.bar(iflags_group, x='iflags', y='count', title= 'Packets by Iflags'))
#fig.show()

In [7]:
#end_group = df.groupby('end-reason').size().reset_index(name = 'count')
#fig = FigureResampler(px.bar(end_group, x='end-reason', y='count', title= 'Packets by Iflags'))
#fig.show()

In [11]:
heatmap = df.pivot_table(index='iflags', columns='end-reason', values='pkt', aggfunc='count')
fig = px.imshow(heatmap, labels={'x': 'End Reason', 'y': 'Iflags', 'color': 'Count'}, title='Iflags VS End Reasons Heatmap')
#fig.show()

In [12]:
fig = px.line(df, x='start-time', y='pkt', title='Time-Series Data from PKT size')
#fig.show()